# Stage 14 — VideoMAE-base + gradual unfreeze + CTC, on FULL FRAMES

Clean encoder comparison against the paper's R3D (Stage 13B) on the **same input paradigm** — full frames, no hand crop, no MediaPipe. The only variable vs the paper is the encoder (VideoMAE-base vs R3D) and the adaptation strategy (gradual unfreezing).

- Full frames resized to 224×224 (no crop), read on-the-fly from the Kaggle WiTA JPGs.
- VideoMAE-base + ConvTranspose1d upsample (T=8→24) + Linear CTC head. **No attention decoder, no landmarks.**
- 4-phase gradual unfreeze: frozen → last-4 blocks → last-8 → full. Phase-conditional LRs.
- AMP fp16 + gradient checkpointing (fits T4 16 GB).
- Checkpoint round-trip via HF Hub for the 2-session split.
- **Test eval exactly once** at the end.

## Settings (right panel)
- Accelerator: **GPU T4 ×1**
- Internet: **ON** (HuggingFace model download + checkpoint push)
- Persistence: Variables and files

## Attach
- `gaurs86/wita-full-english-122signers` (raw WiTA JPGs — same dataset Stage 11/13B used)

## Secrets (Add-ons → Secrets)
- `HF_TOKEN` — HuggingFace write token (for checkpoint round-trip across sessions)

## Honest expectation
Stage 12 (VideoMAE + LoRA + crop + landmarks) hit test 0.59. This tests whether full fine-tuning + full-frame input does better. P(beat paper 0.2924) ≈ 8–12%; realistic landing 0.45–0.55. Primary value: independent encoder data point + ensemble companion to Stage 13B.

## Cell 1 — Install + clone + setup

In [ ]:
%%capture
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.36,<4.50', 'accelerate', 'editdistance',
                'huggingface_hub'], check=True)

import os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b stage13b-paper-replication 'https://github.com/Gaurs86/WiTA-v2.git' '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working/wita_v2')
import torch, transformers
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('transformers', transformers.__version__)

In [ ]:
import os, json, time, random, shutil
import numpy as np
import torch
from torch.utils.data import DataLoader

# Secrets (HF token for cross-session checkpoint round-trip)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    HF_OK = True
except Exception as e:
    print('No HF_TOKEN secret — cross-session resume disabled.', e)
    HF_OK = False

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
device = torch.device('cuda')

from stage14.stage14_lib import (
    CharConverter, WiTAFullFrameDataset, collate_fn, VideoMAE_CTC,
    set_phase, build_optimizer_for_phase, train_phase, evaluate_test,
    VOCAB, T_OUT,
)
print('stage14_lib imported OK')

## Cell 2 — Locate the dataset (hardcoded; no recursive glob)

In [ ]:
DATA_ROOT = '/kaggle/input/datasets/gaurs86/wita-full-english-122signers'
if not os.path.isdir(os.path.join(DATA_ROOT, 'eng_train_lex')):
    cand = None
    for c in os.listdir('/kaggle/input'):
        p1 = os.path.join('/kaggle/input', c)
        if not os.path.isdir(p1):
            continue
        if os.path.isdir(os.path.join(p1, 'eng_train_lex')):
            cand = p1; break
        for c2 in os.listdir(p1):
            p2 = os.path.join(p1, c2)
            if os.path.isdir(p2) and os.path.isdir(os.path.join(p2, 'eng_train_lex')):
                cand = p2; break
        if cand: break
    assert cand, 'eng_train_lex not found — attach gaurs86/wita-full-english-122signers'
    DATA_ROOT = cand
print('DATA_ROOT =', DATA_ROOT)
for split in ('train', 'val', 'test'):
    for subset in ('lex', 'nonlex'):
        d = os.path.join(DATA_ROOT, f'eng_{split}_{subset}')
        print(f'  eng_{split}_{subset:<7s} {"OK" if os.path.isdir(d) else "MISSING"}')

## Cell 3 — Sanity checks (S1 model forward, S4 data, S5 overfit)

In [ ]:
import torch.nn as nn
converter = CharConverter()

# S1 — model forward + memory
m = VideoMAE_CTC(vocab_size=VOCAB).to(device)
x = torch.randn(2, 16, 3, 224, 224, device=device)
with torch.amp.autocast('cuda', dtype=torch.float16):
    y = m(x)
print(f'S1 output {tuple(y.shape)}  (expect [2, {T_OUT}, {VOCAB}])  '
      f'peak mem {torch.cuda.max_memory_allocated()/1e9:.2f} GB')
for ph in (1, 2, 3, 4):
    info = set_phase(m, ph)
    print(f'  phase {ph}: {info["trainable"]/1e6:5.1f}M trainable / {info["total"]/1e6:.1f}M')

# S4 — dataset
val_ds = WiTAFullFrameDataset(DATA_ROOT, 'val', converter, augment=False)
v, t = val_ds[0]
print(f'S4 video {tuple(v.shape)}  target {t.tolist()}  decoded={converter.decode_ctc(t.tolist())!r}  '
      f'label={val_ds.entries[0]["label"]!r}')

# S5 — single-batch overfit (loss should fall fast)
set_phase(m, 4)
opt = build_optimizer_for_phase(m, 4)
ctc = nn.CTCLoss(blank=0, zero_infinity=True)
scaler = torch.amp.GradScaler('cuda')
fb = collate_fn([val_ds[i] for i in range(4)])
vv, yy, yl = fb[0].to(device), fb[1].to(device), fb[2].to(device)
xl = torch.full((4,), T_OUT, dtype=torch.long, device=device)
for step in range(60):
    opt.zero_grad()
    with torch.amp.autocast('cuda', dtype=torch.float16):
        lp = m(vv).float().permute(1, 0, 2).log_softmax(-1)
        loss = ctc(lp, yy, xl, yl)
    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    if step % 15 == 0:
        print(f'  S5 step {step}: loss={loss.item():.4f}')
print(f'S5 final loss {loss.item():.4f}  (should be dropping; pipeline OK if so)')
del m, opt, scaler; torch.cuda.empty_cache()

## Cell 4 — Build datasets + loaders + model

In [ ]:
BATCH = 8
NUM_WORKERS = 2
CKPT = '/kaggle/working/stage14_latest.pt'
HF_REPO = 'gaurs86/stage14-videomae-checkpoint'   # <- create this model repo on HF (private OK)

train_ds = WiTAFullFrameDataset(DATA_ROOT, 'train', converter, augment=True)
val_ds   = WiTAFullFrameDataset(DATA_ROOT, 'val',   converter, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, drop_last=True,
                          persistent_workers=True, prefetch_factor=4)
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn,
                          persistent_workers=True, prefetch_factor=4)

model = VideoMAE_CTC(vocab_size=VOCAB).to(device)
scaler = torch.amp.GradScaler('cuda')
print(f'train={len(train_ds)} val={len(val_ds)}  batch={BATCH}')

## Cell 5 — Resume from HF Hub if a checkpoint exists (session 2)

In [ ]:
start_epoch, best_val_cer = 0, float('inf')
# Pull latest checkpoint from HF (session 2+).
if HF_OK and not os.path.exists(CKPT):
    try:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(HF_REPO, 'stage14_latest.pt', token=os.environ['HF_TOKEN'])
        shutil.copy(p, CKPT)
        print('Pulled checkpoint from HF Hub.')
    except Exception as e:
        print('No HF checkpoint yet (fresh start).', e)

if os.path.exists(CKPT):
    state = torch.load(CKPT, map_location='cuda')
    model.load_state_dict(state['model'])
    scaler.load_state_dict(state['scaler'])
    start_epoch = state['epoch']
    best_val_cer = state['best_val_cer']
    print(f'Resumed at epoch {start_epoch}, best_val_cer={best_val_cer:.4f}')
else:
    print('Fresh start at epoch 0.')

## Cell 6 — Train the gradual-unfreeze phases (with stop criteria)

Resumable: re-running picks up from the checkpoint. Session 1 typically reaches ~Phase 2; session 2 finishes Phases 3–4. Push the checkpoint to HF (Cell 7) before a session ends.

In [ ]:
PHASES = [
    {'phase': 1, 'end': 20, 'stop_above': 0.65},
    {'phase': 2, 'end': 45, 'stop_above': 0.55},
    {'phase': 3, 'end': 75, 'stop_above': 0.45},
    {'phase': 4, 'end': 100, 'stop_above': None},
]

def phase_start_epoch(ph):
    prev = [p['end'] for p in PHASES if p['phase'] < ph['phase']]
    return max(prev) if prev else 0

for ph in PHASES:
    if start_epoch >= ph['end']:
        print(f"Phase {ph['phase']} already complete (start_epoch={start_epoch}).")
        continue
    es = max(start_epoch, phase_start_epoch(ph))
    best_val_cer = train_phase(
        model, train_loader, val_loader, converter, device,
        ph['phase'], es, ph['end'], best_val_cer, CKPT, scaler,
    )
    state = torch.load(CKPT, map_location='cuda')
    vc = state['val_cer']
    if ph['stop_above'] is not None and vc > ph['stop_above']:
        print(f"\nSTOP: phase {ph['phase']} ended val_cer={vc:.4f} > {ph['stop_above']}. "
              f"Consistent with Stage 12's plateau — saving quota.")
        break
    start_epoch = ph['end']
print(f'\nbest_val_cer so far: {best_val_cer:.4f}')

## Cell 7 — Push checkpoint to HF Hub (run before a session ends)

In [ ]:
if HF_OK:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ['HF_TOKEN'])
    api.create_repo(HF_REPO, repo_type='model', exist_ok=True, private=True)
    for fn in ('stage14_latest.pt', 'stage14_latest_best.pt'):
        p = f'/kaggle/working/{fn}'
        if os.path.exists(p):
            api.upload_file(path_or_fileobj=p, path_in_repo=fn,
                            repo_id=HF_REPO, repo_type='model')
            print('pushed', fn)
else:
    print('HF disabled; checkpoint stays in /kaggle/working (lost when session ends).')

## Cell 8 — Test evaluation (RUN ONCE, after Phase 4 completes)

Marker-gated. CTC greedy decode (matches the paper). Pushes the result JSON to HF immediately to timestamp it.

In [ ]:
MARKER = '/kaggle/working/.stage14_test_evaluated'
assert not os.path.exists(MARKER), (
    'Test already evaluated this session. Per the one-shot rule, do not re-run. '
    'Delete the marker only if you intentionally must.')

# Load best-val checkpoint.
best_path = '/kaggle/working/stage14_latest_best.pt'
if not os.path.exists(best_path) and HF_OK:
    from huggingface_hub import hf_hub_download
    p = hf_hub_download(HF_REPO, 'stage14_latest_best.pt', token=os.environ['HF_TOKEN'])
    shutil.copy(p, best_path)
state = torch.load(best_path, map_location='cuda')
model.load_state_dict(state['model']); model.eval()
print(f"Loaded best-val: epoch {state['epoch']}, val_cer={state['val_cer']:.4f}")

test_ds = WiTAFullFrameDataset(DATA_ROOT, 'test', converter, augment=False)
result = evaluate_test(model, test_ds, converter, device,
                       batch_size=4, num_workers=NUM_WORKERS)
result['best_val_cer'] = state['val_cer']
result['best_val_epoch'] = state['epoch']
result['config'] = {'backbone': 'videomae-base', 'input': 'full_frame_224_no_crop',
                    'adaptation': 'gradual_unfreeze_4phase', 'decoder': 'ctc_only',
                    't_in': 16, 't_out': T_OUT, 'vocab': VOCAB}

with open('/kaggle/working/stage14_test_result.json', 'w') as f:
    json.dump(result, f, indent=2)
open(MARKER, 'w').write('done')

print('\n' + '=' * 60)
print('  STAGE 14 TEST HEADLINE (full-frame VideoMAE + CTC)')
print('=' * 60)
print(f"  test_overall_cer : {result['test_overall_cer']:.4f}   (paper 0.2924, S12 0.5913)")
print(f"  test_lex_cer     : {result['test_lex_cer']:.4f}       (paper 0.281)")
print(f"  test_nonlex_cer  : {result['test_nonlex_cer']:.4f}     (paper 0.365)")
print(f"  per_length_cer   : {result['per_length_cer']}")
print('=' * 60)

if HF_OK:
    from huggingface_hub import HfApi
    HfApi(token=os.environ['HF_TOKEN']).upload_file(
        path_or_fileobj='/kaggle/working/stage14_test_result.json',
        path_in_repo='stage14_test_result.json', repo_id=HF_REPO, repo_type='model')
    print('pushed result to HF.')